In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()
name = "Hongduk Ahn"

In [3]:
reader = PdfReader("me/linkedin_HD.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print(linkedin[:500])

   
연락처
mhda80@gmail.com
www.linkedin.com/in/hongduk-
ahn-9a1761149 (LinkedIn)
대표 보유기술
financial analysis
Project modeling
경영 컨설팅
Languages
English (Full Professional)
Certifications
Microsoft Certified: Azure
Fundamentals
Microsoft Certified: Azure Data
Fundamentals
Microsoft Certified: Security,
Compliance, and Identity
Fundamentals
Microsoft Certified: Azure AI
Fundamentals
데이터분석 준전문가(ADsP)
Patents
배터리 교환 장치 및 배터리 교환 시
스템
서버 및 그것의 동작 방법, 배터리 교
환 시스템
Hongduk Ahn
AI Adoption Architect
대한민국 서울
간


In [5]:
with open("me/summary_HD.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(summary)

Hongduk is an AI Adoption Architect at Microsoft Korea, placed through LTIMindtree. He has 16+ years of enterprise experience at LG Group (LG CNS and LG Energy Solution) and 
holds an MBA in Finance from Korea University. He holds two PCT patents related to battery exchange systems. He is currently studying LLM Engineering while pursing to become an AI Enginner that can navigate/bridge the business and technical sector. 


In [6]:
name = "Hongduk Ahn"

system_prompt = f"You are acting as {name}. You are answering questions on behalf of {name} based on the linkedin profile and summary related to {name}'s career, background, skills and experience. Your answers will be faithful, professional and kind. Be professional and engaging. If you don't know the answer say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n"
system_prompt += f"Always stay in character acting as {name}."

In [7]:
system_prompt

"You are acting as Hongduk Ahn. You are answering questions on behalf of Hongduk Ahn based on the linkedin profile and summary related to Hongduk Ahn's career, background, skills and experience. Your answers will be faithful, professional and kind. Be professional and engaging. If you don't know the answer say so.\n\n## Summary:\nHongduk is an AI Adoption Architect at Microsoft Korea, placed through LTIMindtree. He has 16+ years of enterprise experience at LG Group (LG CNS and LG Energy Solution) and \nholds an MBA in Finance from Korea University. He holds two PCT patents related to battery exchange systems. He is currently studying LLM Engineering while pursing to become an AI Enginner that can navigate/bridge the business and technical sector. \n\n## LinkedIn Profile:\n\xa0 \xa0\n연락처\nmhda80@gmail.com\nwww.linkedin.com/in/hongduk-\nahn-9a1761149 (LinkedIn)\n대표 보유기술\nfinancial analysis\nProject modeling\n경영 컨설팅\nLanguages\nEnglish (Full Professional)\nCertifications\nMicrosoft Certif

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [11]:
from pydantic import BaseModel

In [12]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [15]:
evaluator_system_prompt = f"You are an evaluator. You will be given a conversation between an agent and user. You will evaluate whether the answer from the agent is acceptable or not.\
    An acceptable answer is one that is factually correct, relevant to the question, and well-written. You will also provide feedback on how to improve the answer if it is not acceptable."

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n"
evaluator_system_prompt += f"Use the above information to evaluate the following conversation between the user and the agent. Always stay objective and provide constructive feedback."

In [16]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here is the conversation between the user and the agent:\n\n{history}\n\n"
    user_prompt += f"The user's last message was:\n\n{message}\n\n"
    user_prompt += f"The agent's response was:\n\n{reply}\n\n"
    return user_prompt

In [17]:
import os

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

if anthropic_api_key:
    print("Anthropic API key is set.")
else:
    print("Anthropic API key is not set. Please set it in the .env file.")

Anthropic API key is set.


In [18]:
anthropic_base_url = "https://api.anthropic.com/v1"

claude = OpenAI(base_url=anthropic_base_url, api_key=anthropic_api_key)


In [19]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = claude.beta.chat.completions.parse(model="claude-sonnet-4-5", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [21]:
def rerun(reply, message, history, feedback):
     updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\n\n"
     updated_system_prompt += f"## Your previous answer:\n{reply}\n"
     updated_system_prompt += f"## Feedback from answer:\n{feedback}\n"
     messages = [{"role":"system", "content":updated_system_prompt}] + history+ [{"role":"user", "content": message}]
     response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
     return response.choices[0].message.content

In [22]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    reply = response.choices[0].message.content
    
    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Answer is acceptable.")
    else:  
        print("Answer is not acceptable. Rerunning with feedback.")
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Answer is acceptable.
Answer is acceptable.
Answer is not acceptable. Rerunning with feedback.
Answer is acceptable.
